# Klasifikasi Komentar Judol (TF-IDF + SVM) — v3

**Dataset:** komentar YouTube hasil scraping (`text`, `label`; kolom `video_id`/`channel_name` opsional).

**Urutan kerja:** Setup → Normalisasi + masking brand + dedup → Pemeriksaan data → Kelompok kalimat mirip → Split → Training → Perbandingan penanganan ketimpangan kelas → Threshold → Evaluasi (test, ablasi masking, brand/video baru, dataset eksternal) → Simpan model.

**Perubahan dari v2:**
1. **Brand judol di-mask** (`alexis17`, `sgi88`, `pulauwin`, dst. → `brandtoken`). Model belajar konteks kalimat (wd, bonus, jp, gacor, rekomendasi), bukan menghafal nama brand yang berubah-ubah.
2. **Kelompok kalimat mirip dihitung pada teks yang sudah di-mask**, jadi kerangka kalimat yang sama dengan brand berbeda jatuh ke split yang sama.
3. **Evaluasi leave-brand-out**: model diuji pada brand yang tidak ada di data latih. Ini yang paling mendekati komentar nyata.
4. **Ketimpangan kelas tidak diselesaikan dengan resampling.** Data tetap utuh (prevalensi alami), memakai `class_weight="balanced"`, dan ada tabel perbandingan undersampling / SMOTE untuk membuktikannya pada data ini. Metrik utama PR-AUC + precision/recall kelas judol, bukan akurasi.
5. **Threshold diperbaiki**: dihitung dengan bobot prevalensi realistis, `t_low < t_high` dijamin atau zona review dimatikan dengan pesan yang jelas.
6. Evaluasi eksternal memakai threshold yang sama dengan aplikasi.

Jalankan dari atas ke bawah (Run all). Fitur model hanya teks `komentar`; `text_preprocessed` dan kolom metadata (`video_id`, `channel_name`, `tanggal`, `author`, dst.) tidak dipakai sebagai fitur.

## Import Data & Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

FOLDER = "/content/drive/MyDrive/ApkWeb/Dataset"

print("Folder ada:", os.path.exists(FOLDER))
for root, _, files in os.walk(FOLDER):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
import pandas as pd
import numpy as np

# ====== KONFIGURASI: SESUAIKAN ======
DATA_FILE   = "/content/drive/MyDrive/ApkWeb/Dataset/dataset.csv"   # <- ganti nama file jika berbeda
TEXT_COL    = "text"           # teks komentar asli (BUKAN text_preprocessed / komentar_clean)
LABEL_COL   = "label"
VIDEO_COL   = "video_id"       # hanya untuk evaluasi per video, BUKAN fitur model (opsional)
CHANNEL_COL = "channel_name"   # hanya untuk analisis, BUKAN fitur model (opsional)
SEED = 42
# ====================================

if DATA_FILE.lower().endswith((".xlsx", ".xls")):
    df = pd.read_excel(DATA_FILE)
else:
    try:
        df = pd.read_csv(DATA_FILE)
    except UnicodeDecodeError:
        df = pd.read_csv(DATA_FILE, encoding="latin-1")

wajib = [TEXT_COL, LABEL_COL]
hilang = [c for c in wajib if c not in df.columns]
assert not hilang, f"Kolom tidak ditemukan: {hilang}. Kolom yang ada: {list(df.columns)}"
for kol in (VIDEO_COL, CHANNEL_COL):
    if kol not in df.columns:
        print(f"PERINGATAN: kolom {kol} tidak ada, diisi '_na' (evaluasi per video/channel kurang bermakna).")
        df[kol] = "_na"

print("Kolom:", list(df.columns))
print("Jumlah baris:", len(df))
print("\nDistribusi label:")
print(df[LABEL_COL].value_counts(dropna=False))
print("\nContoh data:")
display(df[[TEXT_COL, LABEL_COL]].sample(10, random_state=1))
print("\nNilai kosong:\n", df[[TEXT_COL, LABEL_COL, VIDEO_COL, CHANNEL_COL]].isna().sum())

In [ ]:
print("=== 1. Contoh komentar JUDOL ===")
judol = df[df[LABEL_COL] == 1]
display(judol[[TEXT_COL]].sample(min(15, len(judol)), random_state=1))

print("\n=== 2. Contoh komentar NORMAL ===")
display(df[df[LABEL_COL] == 0][[TEXT_COL]].sample(15, random_state=1))

if "author" in df.columns:
    print("\n=== 3. Seberapa banyak akun yang sama berkomentar? ===")
    per_akun = df.groupby("author").size().sort_values(ascending=False)
    print("Jumlah akun unik:", len(per_akun))
    print("Top 10 akun:\n", per_akun.head(10))
    top = df[df["author"].isin(per_akun.head(10).index)]
    print("\nProporsi judol dari 10 akun teratas:")
    print(top.groupby("author")[LABEL_COL].mean().round(2))

## Normalisasi + Masking Brand + Dedup

`normalizer.py` ditulis sebagai file terpisah supaya aplikasi memakai kode yang sama persis. Fungsi yang dipakai aplikasi: **`prepare(teks)`** = normalisasi + masking brand.

Masking brand: token berpola `3+ huruf + 2-3 angka` (`alexis17`, `sgi88`; kode mobil 2 huruf seperti `ae86`/`rb26` dibiarkan), berakhiran `toto/togel/win` (`agustoto`, `pulauwin`), atau `...4d` diganti `brandtoken`. `maxwin/makswin/mekswin` **tidak** di-mask karena itu kata kunci judol, bukan brand. Model kini melihat "main di brandtoken, wd 5 juta" yang sama untuk semua brand, termasuk brand baru yang belum pernah dilihat.

In [ ]:
import importlib

NORMALIZER_CODE = r"""
import re
import unicodedata

URL_RE   = re.compile(r"(https?://\S+|www\.\S+|\b\S+\.(?:com|net|org|xyz|top|site|online|id|co|me|vip|bet)\b\S*)")
PHONE_RE = re.compile(r"(?:\+?62|0)\s?8[\d\s\-\.]{7,14}\d")
MENTION_RE = re.compile(r"@\S+")
EMOJI_CODE_RE = re.compile(r":[a-z_]+:")
SPACED_RE = re.compile(r"(?<![a-z0-9])(?:[a-z0-9]\s){2,}[a-z0-9](?![a-z0-9])")
LEET = {"0": "o", "1": "i", "3": "e", "4": "a", "5": "s", "7": "t"}
LEET_RE = re.compile(r"(?<=[a-z])[013457](?=[a-z])")
REPEAT_RE = re.compile(r"(.)\1{2,}")

# huruf mirip (Yunani/Sirilik) dan angka dingbat -> Latin/angka biasa
_HOMO_SRC = {"ο": "o", "в": "b", "е": "e", "а": "a", "р": "p",
             "с": "c", "х": "x", "у": "y", "і": "i", "ѕ": "s"}
for _base in (0x2776, 0x2780, 0x278A):
    for _i in range(10):
        _HOMO_SRC[chr(_base + _i)] = str((_i + 1) % 10)
HOMO = str.maketrans(_HOMO_SRC)

def _unspace(m):
    s = m.group(0)
    return re.sub(r"\s", "", s) if re.search(r"[a-z]", s) else s

def normalize(text, full=True):
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = "".join(c for c in text if unicodedata.category(c) not in ("Mn", "Cf"))   # buang tanda gabung + zero-width
    text = text.lower()
    if not full:
        return re.sub(r"\s+", " ", text).strip()
    text = text.translate(HOMO)
    text = MENTION_RE.sub(" ", text)
    text = EMOJI_CODE_RE.sub(" ", text)
    text = URL_RE.sub(" urltoken ", text)
    text = PHONE_RE.sub(" phonetoken ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)       # simbol/pemisah -> spasi
    text = SPACED_RE.sub(_unspace, text)           # gabung huruf tunggal berderet
    for _ in range(2):
        text = LEET_RE.sub(lambda m: LEET[m.group(0)], text)
    text = REPEAT_RE.sub(r"\1\1", text)
    return re.sub(r"\s+", " ", text).strip()

# ---------- masking brand judol ----------
# alexis17, sgi88, mandalika77 (minimal 3 huruf; kode mobil seperti ae86/rb26/cx90 TIDAK di-mask) | agustoto, wisdomtoto, bwtogel | pulauwin (kecuali maxwin/makswin/mekswin)
BRAND_RE = re.compile(
    r"\b(?:[a-z]{3,}\d{2,3}(?!\d)"
    r"|(?:[a-z]{3,}(?:toto|togel)"
    r"|(?!(?:max|maks|meks|mex)win\b)[a-z]{3,}win)\b)")
# pola "pas4d": harus diganti SEBELUM leet-normalisasi mengubah 4 -> a
BRAND4D_RE = re.compile(r"\b[a-z]{2,}\s?4d\b")
_PREFIX_RE = re.compile(r"^(?:di|sibuk|dari|ke|se|pake|pakai)(?=[a-z]{3,}\d)")

def mask_brand(text_norm):
    return BRAND_RE.sub(" brandtoken ", text_norm).strip()

def _pre(text):
    text = unicodedata.normalize("NFKC", text)
    text = "".join(c for c in text if unicodedata.category(c) not in ("Mn", "Cf"))
    return text.lower()

def brand_of(text):
    # menerima teks MENTAH; mengembalikan nama brand pertama atau "_none"
    if not isinstance(text, str):
        return "_none"
    raw = _pre(text)
    m4 = BRAND4D_RE.search(raw)
    if m4:
        return m4.group(0)
    m = BRAND_RE.search(normalize(raw))
    if not m:
        return "_none"
    b = _PREFIX_RE.sub("", m.group(0))
    return "alexis17" if b in ("lexis17", "alexis17") else b

def prepare(text):
    # PINTU MASUK model (dipakai notebook DAN aplikasi): normalisasi + masking brand
    if not isinstance(text, str):
        return ""
    raw = BRAND4D_RE.sub(" brandtoken ", _pre(text))
    return re.sub(r"\s+", " ", mask_brand(normalize(raw))).strip()
"""
with open("normalizer.py", "w", encoding="utf-8") as f:
    f.write(NORMALIZER_CODE)

import normalizer
importlib.reload(normalizer)
from normalizer import normalize, prepare, mask_brand, brand_of

# ===== Bersihkan + dedup =====
LABEL_MAP = {0: 0, 1: 1}
MASK_ANGKA = True   # True = "wd 5 juta" dan "wd 8 juta" dianggap template yang sama

data = df[[TEXT_COL, LABEL_COL, VIDEO_COL, CHANNEL_COL]].copy()
data.columns = ["text_raw", "label_raw", "video_id", "channel"]
data = data.dropna(subset=["text_raw", "label_raw"])
data["y"] = data["label_raw"].map(LABEL_MAP)
assert data["y"].notna().all(), "Ada label di luar LABEL_MAP, cek value_counts di cell data."
data["y"] = data["y"].astype(int)
data["text_raw"] = data["text_raw"].astype(str)
data["text_norm"] = data["text_raw"].map(normalize)       # hanya normalisasi
data["text_mask"] = data["text_raw"].map(prepare)         # normalisasi + brand di-mask (INI yang dipakai model)
data = data[data["text_norm"].str.len() > 0]

data["dedup_key"] = data["text_norm"].str.replace(r"\d+", "0", regex=True) if MASK_ANGKA else data["text_norm"]
sebelum = data["y"].value_counts().sort_index()

konflik = data.groupby("dedup_key")["y"].nunique()
konflik = konflik[konflik > 1].index
print("Teks dengan label bertentangan (dibuang):", len(konflik))
data = data[~data["dedup_key"].isin(konflik)]

data["n_ulang"] = data.groupby("dedup_key")["dedup_key"].transform("size")
template_teratas = (data[data["y"] == 1]
                    .drop_duplicates("dedup_key")
                    .sort_values("n_ulang", ascending=False)
                    .head(10)[["text_raw", "n_ulang"]])
data = data.drop_duplicates("dedup_key").reset_index(drop=True)
data["brand"] = data["text_raw"].map(brand_of)

sesudah = data["y"].value_counts().sort_index()
print("\n=== Jumlah baris per kelas ===")
print(pd.DataFrame({"sebelum": sebelum, "sesudah": sesudah,
                    "tersisa_%": (sesudah / sebelum * 100).round(1)}))
print("\nProporsi kelas sesudah dedup (ini prevalensi ALAMI, jangan diubah dengan resampling):")
print(data["y"].value_counts(normalize=True).round(3))
print("\n10 template judol yang paling sering diulang:")
display(template_teratas.reset_index(drop=True))

print("\nBrand judol terdeteksi (top 15):")
print(data[data["y"] == 1]["brand"].value_counts().head(15).to_string())
print("Judol tanpa brand terdeteksi:", f"{(data[data['y'] == 1]['brand'] == '_none').mean():.1%}")
print("Normal yang kena pola brand (mis. ae86, cx90):", f"{(data[data['y'] == 0]['brand'] != '_none').mean():.2%}")

## Pemeriksaan Kualitas Dataset

In [ ]:
# 1. Label vs predicted_label (keluaran model lain, TIDAK dipakai sebagai fitur)
if "predicted_label" in df.columns:
    print(pd.crosstab(df[LABEL_COL], df["predicted_label"]))
    beda = df[df[LABEL_COL] != df["predicted_label"]]
    print("Baris yang berbeda:", len(beda))

# 2. Sebaran label per video
tab = pd.crosstab(df[VIDEO_COL], df[LABEL_COL])
print("\nJumlah video:", len(tab))
if tab.shape[1] == 2:
    print("Video yang hanya punya judol :", int(((tab[1] > 0) & (tab[0] == 0)).sum()))
    print("Video yang hanya punya normal:", int(((tab[0] > 0) & (tab[1] == 0)).sum()))
    display(tab.sort_values(1, ascending=False).head(10))

# 3. Sebaran label per channel
print("\nPersentase judol per channel:")
display(df.groupby(CHANNEL_COL)[LABEL_COL].agg(jumlah="size", persen_judol="mean")
        .round(3).sort_values("jumlah", ascending=False))

# 4. Format tanggal vs label (data dikumpulkan dalam beberapa batch)
if "tanggal" in df.columns:
    iso = df["tanggal"].astype(str).str.contains("-")
    print("\nPersentase judol berdasarkan format kolom tanggal:")
    print(df.groupby(iso)[LABEL_COL].mean().round(3).rename({True: "format ISO", False: "format angka"}))
    print("-> kolom tanggal tidak dipakai sebagai fitur.")

# 5. Uji pintasan gaya penulisan
s = df[TEXT_COL].astype(str)
fitur = pd.DataFrame({
    "spasi_sebelum_tanda": s.str.contains(r"\s[,.!?]"),
    "ada_emoji": s.str.contains(r"[\U0001F300-\U0001FAFF\u2600-\u27BF]"),
    "ada_mention": s.str.contains(r"@\w"),
    "ada_unicode_gaya": s.str.contains(r"[\U0001D400-\U0001D7FF]"),
    "jumlah_kata": s.str.split().str.len(),
    "label": df[LABEL_COL],
})
print("\nRata-rata fitur gaya per kelas:")
print(fitur.groupby("label").mean().round(3).T)
print("Perbedaan besar di sini = pintasan potensial. Fitur ini sengaja TIDAK ditambahkan ke model.")

In [ ]:
# Seberapa besar kontrol brand pada data? (judol setelah normalisasi dan setelah masking)
j = data[data["y"] == 1]
print("Judol unik (normalisasi)             :", j["text_norm"].nunique())
print("Judol unik (normalisasi + mask brand):", j["text_mask"].str.replace(r"\d+", "0", regex=True).nunique())
print("-> selisih besar = banyak komentar judol hanyalah kerangka sama dengan brand berbeda.")

contoh = [
    "𝗚𝗔𝗖𝗢𝗥𝟳𝟳 maxwin hari ini!!!",
    "a l e x i s 1 7 bikin ketagihan",
    "Langsung ke 𝙿 𝚄 𝙻 𝙰 𝚄 𝚆 𝙸 𝙽 🎮",
    "NewMember :star:PAS4D Pasti Jp",
    "100 gratis wisdomtoto",
    "makasih bang, kemarin wd 5 juta di 𝗦𝗚𝗜𝟴𝟴",
    "Ada mobil akang mv gt86",
    "chat wa 0812-3456-7890 sekarang",
    "klik https://situs-judi.xyz/daftar",
    "mantaaaap bang videonya",
]
for c in contoh:
    print(f"{c}\n  -> {prepare(c)}\n")

## Kelompok Kalimat Mirip (Anti-Kebocoran)

Kerangka kalimat yang sama dengan brand berbeda harus berada di split yang sama. Kemiripan dihitung pada **`text_mask`** (brand sudah di-mask), jadi `cakep kali weton88 bravo` dan `cakep kali sgi88 bravo` pasti satu kelompok. Dataset besar: proses dibagi per 300 baris agar memori Colab tidak habis.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

THR_KANDIDAT = [0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]
MAKS_UKURAN_GRUP = 0.03   # grup terbesar tidak boleh lebih dari 3% seluruh baris
CHUNK = 300

vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)
M = vec.fit_transform(data["text_mask"]).astype(np.float32)
n = M.shape[0]
T0 = min(THR_KANDIDAT)

r_list, c_list, s_list = [], [], []
MT = M.T.tocsc()
for s in range(0, n, CHUNK):
    sim = (M[s:s + CHUNK] @ MT).tocoo()
    keep = (sim.data >= T0) & ((sim.row + s) < sim.col)     # pasangan i<j saja
    r_list.append(sim.row[keep] + s)
    c_list.append(sim.col[keep])
    s_list.append(sim.data[keep])
rows = np.concatenate(r_list); cols = np.concatenate(c_list); sims = np.concatenate(s_list)
print(f"Pasangan komentar mirip (similarity >= {T0}): {len(rows):,}\n")

pilih, terakhir = None, None
for thr in THR_KANDIDAT:
    k = sims >= thr
    G = csr_matrix((np.ones(int(k.sum())), (rows[k], cols[k])), shape=(n, n))
    jml, lab = connected_components(G, directed=False)
    terbesar = int(np.bincount(lab).max())
    print(f"THR = {thr:.2f} -> {jml:5d} grup | grup terbesar = {terbesar:5d} baris ({terbesar / n:.1%})")
    terakhir = (thr, lab)
    if pilih is None and terbesar / n <= MAKS_UKURAN_GRUP:
        pilih = (thr, lab)

THR_PAKAI, grup_id = pilih if pilih is not None else terakhir
data["group"] = grup_id
ukuran = data.groupby("group").size().sort_values(ascending=False)
print(f"\nThreshold dipakai: {THR_PAKAI:.2f} | jumlah grup: {data['group'].nunique()} dari {len(data)} baris")
print("Contoh isi grup terbesar:")
display(data[data["group"] == ukuran.index[0]][["text_raw", "y"]].head(8))

## Split Train/Val/Test (per Kelompok)

Split **tidak** diseimbangkan: val dan test mempertahankan prevalensi alami. Menyeimbangkan data uji membuat precision tampak lebih baik dari kenyataan.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)   # ~14% per fold
folds = [te for _, te in sgkf.split(data, data["y"], groups=data["group"])]
test_idx, val_idx = folds[0], folds[1]
train_idx = np.setdiff1d(np.arange(len(data)), np.concatenate([test_idx, val_idx]))

train_df = data.iloc[train_idx].reset_index(drop=True)
val_df   = data.iloc[val_idx].reset_index(drop=True)
test_df  = data.iloc[test_idx].reset_index(drop=True)

for nama, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{nama:5s}: {len(d):6d} baris | judol = {int(d['y'].sum()):5d} ({d['y'].mean():.1%}) | video = {d['video_id'].nunique()}")

g_tr, g_va, g_te = set(train_df["group"]), set(val_df["group"]), set(test_df["group"])
assert not (g_tr & g_va) and not (g_tr & g_te) and not (g_va & g_te), "Ada kelompok yang bocor antar split"
print("\nTidak ada kelompok yang sama antar split.")

X_train, y_train = train_df["text_mask"], train_df["y"]
X_val,   y_val   = val_df["text_mask"],   val_df["y"]
X_test,  y_test  = test_df["text_mask"],  test_df["y"]

# kelompok untuk evaluasi leave-brand-out: judol -> brand (atau kelompok kalimat bila tak ada brand),
# normal -> video (bila kolom video ada; bila tidak, kelompok kalimat)
data["lbo_group"] = np.where(
    data["y"] == 1,
    np.where(data["brand"] != "_none", "b_" + data["brand"], "g_" + data["group"].astype(str)),
    np.where(data["video_id"].nunique() > 1, "v_" + data["video_id"].astype(str), "g_" + data["group"].astype(str)))

## Training

In [ ]:
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

BEST_NAME = "SVM (kalibrasi)"

def make_features():
    return FeatureUnion([
        ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=2,
                                 sublinear_tf=True, max_features=50000)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2,
                                 sublinear_tf=True, max_features=100000)),
    ])

def make_models(C=1.0, cv=5):
    return {
        BEST_NAME: CalibratedClassifierCV(
            LinearSVC(C=C, class_weight="balanced", max_iter=5000), cv=cv, method="sigmoid")
    }

def build(clf):
    return Pipeline([("tfidf", make_features()), ("clf", clf)])

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score

# Tuning C dengan CV per kelompok. Metrik: average_precision (PR-AUC), cocok untuk kelas judol yang jarang.
cv_grup = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
hasil_C = {}
for C in [0.1, 0.5, 1.0, 2.0, 5.0]:
    pipe = build(LinearSVC(C=C, class_weight="balanced", max_iter=5000))
    skor = cross_val_score(pipe, X_train, y_train, cv=cv_grup,
                           groups=train_df["group"], scoring="average_precision", n_jobs=-1)
    hasil_C[C] = skor.mean()
    print(f"C = {C:<4} PR-AUC CV = {skor.mean():.4f} ± {skor.std():.4f}")

BEST_C = max(hasil_C, key=hasil_C.get)
print("\nC terbaik:", BEST_C)

## Evaluasi Data Validasi

In [ ]:
from sklearn.metrics import classification_report, average_precision_score

cal_cv = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
              .split(X_train, y_train, groups=train_df["group"]))

model = build(make_models(C=BEST_C, cv=cal_cv)[BEST_NAME])
model.fit(X_train, y_train)

p_val = model.predict_proba(X_val)[:, 1]
print(f"PR-AUC validasi: {average_precision_score(y_val, p_val):.4f}")
print(classification_report(y_val, (p_val >= 0.5).astype(int),
                            target_names=["bukan", "judol"], digits=4))

## Perbandingan Penanganan Ketimpangan Kelas

Pertanyaan: apakah data perlu diseimbangkan dengan **undersampling** komentar normal atau **SMOTE** untuk judol? Cell ini mengujinya pada split yang sama (resampling hanya pada data **train**; val tidak disentuh). SMOTE ditulis manual untuk matriks TF-IDF sparse (tanpa `imbalanced-learn`).

Cara membaca: bandingkan **PR-AUC** (tidak tergantung threshold) dan **FP** (normal yang salah dilabeli judol). Pilih metode yang PR-AUC-nya terbaik. Selisih kecil (< ±0,01) berarti metode itu tidak memberi manfaat nyata.

In [ ]:
from scipy.sparse import vstack, diags
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import precision_score, recall_score, f1_score

def smote_sparse(X, y, k=5, seed=SEED):
    rng = np.random.RandomState(seed)
    pos = np.where(y == 1)[0]; neg = np.where(y == 0)[0]
    butuh = len(neg) - len(pos)
    Xp = X[pos]
    nn = NearestNeighbors(n_neighbors=min(k + 1, len(pos)), metric="cosine").fit(Xp)
    idx = nn.kneighbors(Xp, return_distance=False)[:, 1:]
    a = rng.randint(0, len(pos), butuh)
    b = idx[a, rng.randint(0, idx.shape[1], butuh)]
    lam = rng.rand(butuh)
    Xs = diags(1 - lam) @ Xp[a] + diags(lam) @ Xp[b]
    return vstack([X, Xs]).tocsr(), np.r_[y, np.ones(butuh, dtype=int)]

def undersample(X, y, rasio=1, seed=SEED):
    rng = np.random.RandomState(seed)
    pos = np.where(y == 1)[0]; neg = np.where(y == 0)[0]
    pilih_neg = rng.choice(neg, min(len(neg), len(pos) * rasio), replace=False)
    idx = np.r_[pos, pilih_neg]
    return X[idx], y[idx]

fe = make_features().fit(X_train)
Xtr_f, Xva_f = fe.transform(X_train), fe.transform(X_val)
ytr, yva = y_train.to_numpy(), y_val.to_numpy()

baris = []
for nama in ["tanpa penyeimbangan", "class_weight='balanced'", "undersample normal 1:1",
             "undersample normal 3:1", "SMOTE judol -> 1:1"]:
    X_, y_, cw = Xtr_f, ytr, None
    if nama.startswith("class_weight"):
        cw = "balanced"
    elif nama == "undersample normal 1:1":
        X_, y_ = undersample(Xtr_f, ytr, 1)
    elif nama == "undersample normal 3:1":
        X_, y_ = undersample(Xtr_f, ytr, 3)
    elif nama.startswith("SMOTE"):
        X_, y_ = smote_sparse(Xtr_f, ytr)
    m = LinearSVC(C=BEST_C, class_weight=cw, max_iter=5000).fit(X_, y_)
    sc = m.decision_function(Xva_f); pr = (sc > 0).astype(int)
    baris.append({"Metode": nama, "baris latih": X_.shape[0],
                  "PR-AUC": average_precision_score(yva, sc),
                  "Precision": precision_score(yva, pr, zero_division=0),
                  "Recall": recall_score(yva, pr),
                  "F1": f1_score(yva, pr, zero_division=0),
                  "FP": int(((pr == 1) & (yva == 0)).sum()),
                  "FN": int(((pr == 0) & (yva == 1)).sum())})
perbandingan_imbalance = pd.DataFrame(baris).round(4)
display(perbandingan_imbalance)
print("Data tidak diubah di atas; model final tetap memakai data utuh + class_weight='balanced' (kecuali tabel ini menunjukkan keunggulan nyata).")

## Adjust Threshold

Dua ambang: di atas `t_high` = otomatis **judol**, di bawah `t_low` = otomatis **bukan**, di antaranya = **review** manual. Precision/recall dihitung dengan **bobot prevalensi realistis** (`PREV_DEPLOY`), karena komentar nyata di channel pengguna jauh lebih jarang judol daripada data latih. Set `PREV_DEPLOY = None` untuk memakai prevalensi data validasi apa adanya.

In [ ]:
from sklearn.metrics import precision_recall_curve

# ====== TARGET: SESUAIKAN ======
P_TARGET    = 0.95   # precision minimum agar komentar otomatis berlabel "judol"
R_TARGET    = 0.97   # recall minimum; komentar di bawah t_low dianggap "bukan"
PREV_DEPLOY = 0.05   # perkiraan proporsi judol di komentar nyata (None = pakai data validasi)
# ===============================

yv = y_val.to_numpy()
w = np.ones(len(yv))
if PREV_DEPLOY:
    n_pos, n_neg = int(yv.sum()), int((yv == 0).sum())
    w = np.where(yv == 1, 1.0, (1 - PREV_DEPLOY) / PREV_DEPLOY * n_pos / n_neg)

prec, rec, thr = precision_recall_curve(yv, p_val, sample_weight=w)
prec, rec = prec[:-1], rec[:-1]

ok_high = np.where(prec >= P_TARGET)[0]
ok_low  = np.where(rec >= R_TARGET)[0]
t_high = float(thr[ok_high[0]]) if len(ok_high) else 0.5
t_low  = float(thr[ok_low[-1]]) if len(ok_low) else 0.5

if not len(ok_high) or not len(ok_low):
    print("PERINGATAN: target precision/recall tidak tercapai di data validasi. Longgarkan P_TARGET/R_TARGET.")
if t_low >= t_high:
    print(f"Target precision dan recall tercapai pada ambang yang sama (t_low={t_low:.3f} >= t_high={t_high:.3f}).")
    print("Zona 'review' tidak diperlukan: ambang tunggal dipakai.")
    t_low = t_high
else:
    print(f"t_low = {t_low:.3f} | t_high = {t_high:.3f} (zona review ada di antaranya)")

def zona(p):
    return np.where(p >= t_high, "judol", np.where(p <= t_low, "bukan", "review"))

z = pd.Series(zona(p_val))
print("\nSebaran zona di data validasi (prevalensi data):")
print(z.value_counts())

## Evaluasi Data Test

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

y_t = y_test.to_numpy()
p_test = model.predict_proba(X_test)[:, 1]
pred_test = (p_test >= 0.5).astype(int)

print(f"PR-AUC test: {average_precision_score(y_t, p_test):.4f}")
print("=== Evaluasi di data TEST (threshold 0.5) ===")
print(classification_report(y_t, pred_test, target_names=["bukan", "judol"], digits=4))

cm = confusion_matrix(y_t, pred_test)
ConfusionMatrixDisplay(cm, display_labels=["bukan", "judol"]).plot(cmap="Blues")
plt.title(f"Confusion Matrix - {BEST_NAME}")
plt.show()

zt = zona(p_test)
auto = zt != "review"
pred_auto = (zt[auto] == "judol").astype(int)
print("\n=== Skema tiga zona (ambang dari validasi) di data TEST ===")
print(pd.Series(zt).value_counts())
print(f"Komentar diputuskan otomatis   : {auto.mean():.1%}")
print(f"Precision judol (zona judol)   : {precision_score(y_t[auto], pred_auto, zero_division=0):.4f}")
print(f"Recall judol (di zona otomatis): {recall_score(y_t[auto], pred_auto, zero_division=0):.4f}")
print(f"Judol di zona review           : {int(y_t[~auto].sum())} dari {int(y_t.sum())}")

## Ablasi: Efek Masking Brand

Model dan split yang sama, hanya teks masukan yang berbeda. Di data uji ini (brand yang sama muncul di train dan test) selisihnya mungkin kecil. Efek sebenarnya terlihat pada cell berikutnya, yang menguji brand yang belum pernah dilihat.

In [ ]:
def eval_variant(kolom, nama):
    Xtr, Xte = train_df[kolom], test_df[kolom]
    m = build(LinearSVC(C=BEST_C, class_weight="balanced", max_iter=5000)).fit(Xtr, y_train)
    sc = m.decision_function(Xte); pr = (sc > 0).astype(int)
    return {"Teks masukan": nama, "PR-AUC": average_precision_score(y_t, sc),
            "Precision": precision_score(y_t, pr), "Recall": recall_score(y_t, pr), "F1": f1_score(y_t, pr)}

abl = pd.DataFrame([
    eval_variant("text_norm", "Normalisasi saja (brand utuh)"),
    eval_variant("text_mask", "Normalisasi + brand di-mask"),
]).round(4)
display(abl)

## Evaluasi Generalisasi: Brand dan Video yang Belum Pernah Dilihat

Leave-brand-out: komentar judol dikelompokkan per brand (atau per kerangka kalimat bila tanpa brand), komentar normal per video. Pada tiap fold, semua komentar yang mirip dengan data uji dibuang dari data latih. Ini evaluasi paling ketat dan paling dekat dengan komentar nyata. Dua varian dibandingkan: brand utuh vs brand di-mask.

In [ ]:
from sklearn.model_selection import GroupKFold

def lbo(kolom):
    hasil = []
    for k, (tr, te) in enumerate(GroupKFold(n_splits=5).split(data, data["y"], groups=data["lbo_group"]), 1):
        grup_uji = set(data["group"].iloc[te])
        tr_bersih = tr[~data["group"].iloc[tr].isin(grup_uji).to_numpy()]
        m = build(LinearSVC(C=BEST_C, class_weight="balanced", max_iter=5000)).fit(
            data[kolom].iloc[tr_bersih], data["y"].iloc[tr_bersih])
        yt = data["y"].iloc[te].to_numpy()
        sc = m.decision_function(data[kolom].iloc[te]); pr = (sc > 0).astype(int)
        hasil.append({"Fold": k, "Baris uji": len(te), "Judol uji": int(yt.sum()),
                      "Brand uji": data.loc[data.index[te], "brand"][data["y"].iloc[te] == 1].nunique(),
                      "Train dibuang": int(len(tr) - len(tr_bersih)),
                      "PR-AUC": average_precision_score(yt, sc),
                      "Precision": precision_score(yt, pr, zero_division=0),
                      "Recall": recall_score(yt, pr, zero_division=0),
                      "F1": f1_score(yt, pr, zero_division=0)})
    return pd.DataFrame(hasil).round(4)

res_lbo_raw  = lbo("text_norm")
res_video    = lbo("text_mask")
print("=== Brand utuh (tanpa masking) ===");  display(res_lbo_raw)
print("=== Brand di-mask (dipakai model) ==="); display(res_video)
print(f"Rata-rata F1  : tanpa mask {res_lbo_raw['F1'].mean():.4f} | di-mask {res_video['F1'].mean():.4f}")
print(f"Rata-rata PR-AUC: tanpa mask {res_lbo_raw['PR-AUC'].mean():.4f} | di-mask {res_video['PR-AUC'].mean():.4f}")
print("Angka inilah yang sebaiknya dilaporkan sebagai kemampuan generalisasi, bukan skor test di atas.")

## Evaluasi Eksternal: Dataset Live Chat Jogja

Dataset kedua tidak dipakai untuk training. Isinya spam asli dengan brand lain dan komentar normal yang menyebut slot/judol (*hard negative*). Prevalensi judol-nya sangat rendah, jadi **precision** adalah angka yang paling penting. Lewati dengan `RUN_EKSTERNAL = False` bila file belum ada.

In [ ]:
RUN_EKSTERNAL = True
DATA_JOGJA  = "/content/drive/MyDrive/ApkWeb/Dataset/youtube_chat_jogja_clean.csv"   # <- sesuaikan
TEXT_JOGJA  = "message"
LABEL_JOGJA = "label"

hasil_ext = None
if RUN_EKSTERNAL and os.path.exists(DATA_JOGJA):
    ext = pd.read_csv(DATA_JOGJA)[[TEXT_JOGJA, LABEL_JOGJA]].dropna()
    ext.columns = ["text_raw", "y"]
    ext["text_raw"] = ext["text_raw"].astype(str)
    ext["y"] = ext["y"].astype(int)
    ext["text_norm"] = ext["text_raw"].map(normalize)
    ext["text_mask"] = ext["text_raw"].map(prepare)
    ext = ext[ext["text_norm"].str.len() > 0]
    ext["key"] = ext["text_norm"].str.replace(r"\d+", "0", regex=True)
    ext = ext.drop_duplicates("key")
    ext = ext[~ext["key"].isin(set(data["dedup_key"]))].reset_index(drop=True)

    y_e = ext["y"].to_numpy()
    p_e = model.predict_proba(ext["text_mask"])[:, 1]
    print(f"Data eksternal (unik): {len(ext)} baris | judol = {int(y_e.sum())} ({y_e.mean():.1%}) | normal = {int((y_e == 0).sum())}")
    print(f"PR-AUC eksternal: {average_precision_score(y_e, p_e):.4f}")

    for nama_thr, thr_e in [("threshold 0.5", 0.5), (f"t_high dari validasi ({t_high:.3f})", t_high)]:
        pred_e = (p_e >= thr_e).astype(int)
        print(f"\n--- {nama_thr} ---")
        print(classification_report(y_e, pred_e, target_names=["bukan", "judol"], digits=4, zero_division=0))
        print("Confusion matrix [[TN, FP], [FN, TP]]:\n", confusion_matrix(y_e, pred_e))

    pred_e = (p_e >= t_high).astype(int)
    kw = r"slot|toto|gacor|judi|maxwin|freebet|bonus|togel|chip"
    jebakan = ext["text_raw"].str.lower().str.contains(kw, regex=True).to_numpy() & (y_e == 0)
    print(f"\nKomentar normal yang menyebut kata judol: {int(jebakan.sum())} | salah dilabeli judol: {int(((pred_e == 1) & jebakan).sum())}")

    print("\nJudol yang LOLOS (false negative):")
    display(ext[(y_e == 1) & (pred_e == 0)][["text_raw"]].head(15))
    print("Normal yang SALAH dilabeli judol (false positive):")
    display(ext[(y_e == 0) & (pred_e == 1)][["text_raw"]].head(15))

    hasil_ext = {
        "n": int(len(ext)), "n_judol": int(y_e.sum()),
        "pr_auc": float(average_precision_score(y_e, p_e)),
        "threshold": float(t_high),
        "precision": float(precision_score(y_e, pred_e, zero_division=0)),
        "recall": float(recall_score(y_e, pred_e, zero_division=0)),
        "f1": float(f1_score(y_e, pred_e, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_e, pred_e).tolist(),
    }
else:
    print("Evaluasi eksternal dilewati (RUN_EKSTERNAL=False atau file tidak ditemukan).")

## Save Model

Aplikasi **wajib** memanggil `prepare(teks)` dari `normalizer.py` sebelum `predict_proba`, dan memakai `t_low` / `t_high` dari `metrics.json`.

In [ ]:
import joblib, json, sklearn, shutil
from datetime import datetime

OUT_DIR = "/content/drive/MyDrive/ApkWeb/model"
os.makedirs(OUT_DIR, exist_ok=True)

joblib.dump(model, f"{OUT_DIR}/judol_model.joblib", compress=3)

metrics = {
    "model_name": "TF-IDF (word + char) + LinearSVC terkalibrasi, brand di-mask",
    "params": {"C": float(BEST_C), "class_weight": "balanced", "calibration": "sigmoid, cv=5 (per kelompok)",
               "resampling": "tidak dipakai (prevalensi alami)"},
    "input": "prepare(teks) dari normalizer.py (normalisasi + masking brand)",
    "trained_at": datetime.now().isoformat(timespec="seconds"),
    "sklearn_version": sklearn.__version__,
    "dataset": {
        "baris_mentah": int(len(df)),
        "baris_setelah_dedup": int(len(data)),
        "judol": int(data["y"].sum()),
        "normal": int((data["y"] == 0).sum()),
        "prevalensi_judol": float(data["y"].mean()),
        "jumlah_kelompok": int(data["group"].nunique()),
        "threshold_kemiripan_kelompok": float(THR_PAKAI),
    },
    "split": "StratifiedGroupKFold per kelompok kalimat mirip (7 fold: 5 train, 1 val, 1 test)",
    "n_train": int(len(X_train)), "n_val": int(len(X_val)), "n_test": int(len(X_test)),
    "thresholds": {"t_low": t_low, "t_high": t_high, "p_target": P_TARGET,
                   "r_target": R_TARGET, "prevalensi_bobot": PREV_DEPLOY},
    "test": {
        "pr_auc": float(average_precision_score(y_t, p_test)),
        "precision": float(precision_score(y_t, pred_test)),
        "recall": float(recall_score(y_t, pred_test)),
        "f1": float(f1_score(y_t, pred_test)),
        "confusion_matrix": cm.tolist(),
    },
    "perbandingan_imbalance": perbandingan_imbalance.to_dict(orient="records"),
    "ablasi_masking_brand": abl.to_dict(orient="records"),
    "leave_brand_out": {"tanpa_mask": res_lbo_raw.to_dict(orient="records"),
                        "di_mask": res_video.to_dict(orient="records")},
}
if hasil_ext is not None:
    metrics["eksternal_live_chat_jogja"] = hasil_ext

with open(f"{OUT_DIR}/metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False,
              default=lambda o: o.item() if hasattr(o, "item") else str(o))

shutil.copy("normalizer.py", f"{OUT_DIR}/normalizer.py")

size_mb = os.path.getsize(f"{OUT_DIR}/judol_model.joblib") / 1e6
print(f"Ukuran model: {size_mb:.2f} MB | tersimpan di {OUT_DIR}")
print("sklearn:", sklearn.__version__)

## Uji pada Komentar Nyata

Tambahkan 20-50 komentar nyata dari channel-mu beserta label manual. Ini ujian paling jujur sebelum model dipakai di aplikasi.

In [ ]:
uji_nyata = [
    ("makasih bang, kemarin wd 5 juta di 𝗚𝗔𝗖𝗢𝗥𝟳𝟳", 1),
    ("chat wa 085326697570 bonus new member 100%", 1),
    ("100 gratis wisdomtoto", 1),
    ("NewMember :star:PAS4D Pasti Jp SUDAH PASTI", 1),
    ("main di 𝐀𝐋𝐄𝐗𝐈𝐒𝟏𝟕 gampang wd, modal receh maxwin", 1),
    ("gacor banget videonya", 0),
    ("game slot di hp aku lag terus, ada solusi?", 0),
    ("Mobilnya mantap, mesin RB26 emang juara", 0),
    ("Bahas judi online kapan bang? bahaya banget buat anak muda", 0),
    ("Ada mobil akang mv gt86", 0),
]
Xn = [prepare(t) for t, _ in uji_nyata]
yn = [y for _, y in uji_nyata]
pn = model.predict_proba(Xn)[:, 1]

hasil = pd.DataFrame({
    "komentar": [t for t, _ in uji_nyata],
    "label_manual": yn,
    "skor_judol": pn.round(3),
    "zona": zona(pn),
})
display(hasil)

## Download Model

In [ ]:
from google.colab import files
for nama_file in ["judol_model.joblib", "metrics.json", "normalizer.py"]:
    files.download(f"{OUT_DIR}/{nama_file}")